# PROJECT AKHIR PENGANTAR DATA SAINS (PDS)

**Disusun Oleh:**  
* **Nama:** Siti Fadilah Nurkhotimah  
* **NIM:** 1314623019  
* **Kelas:** Statistika 2023 B  
* **Program Studi:** S1 Statistika, Universitas Negeri Jakarta (UNJ)  

**Dosen Pengampu:**  
* Endang Yuliani, S.Mat., M.Si.  
* Faroh Ladayya, M.Si.  

---
### **Judul:**
### **ANALISIS KOMPARATIF LOGISTIC REGRESSION DAN DECISION TREE UNTUK MEMPREDIKSI KESELAMATAN PENUMPANG KAPAL RMS (ROYAL MAIL SHIP) TITANIC**
---

## 1. Pengambilan Data & Import Library

In [ ]:
# Import Library Manipulasi & Visualisasi Data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Import Library Preprocessing & Modeling
from sklearn.model_selection import train_test_split, KFold, cross_val_score, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree

# Import Library Metrik Evaluasi
from sklearn.metrics import (
    confusion_matrix, 
    classification_report, 
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score, 
    roc_curve, 
    auc
)

import warnings
warnings.filterwarnings('ignore')

print("Semua library berhasil diimpor!")

### Membaca Dataset Titanic (Data Latih / Train) & Data Prediksi (Data Uji / Test)

In [ ]:
# Membaca dataset latih
try:
    data_titanic = pd.read_excel("data_titanic.xlsx")
except:
    data_titanic = pd.read_csv("data_titanic.csv")

# Membaca dataset prediksi (data uji tanpa label)
try:
    data_prediksi = pd.read_excel("data_prediksi.xlsx")
except:
    data_prediksi = pd.read_csv("data_prediksi.csv")

print("--- Data Latih (data_titanic) ---")
display(data_titanic.head())

print("\n--- Informasi Struktur Data Latih ---")
data_titanic.info()

In [ ]:
print("--- Data Prediksi (data_prediksi) ---")
display(data_prediksi.head())

print("\n--- Informasi Struktur Data Prediksi ---")
data_prediksi.info()

## 2. Analisis Data Eksploratif (EDA)

In [ ]:
# Bar Chart Jumlah Penumpang Berdasarkan Jenis Kelamin
count_sex = data_titanic['Sex'].value_counts()
index_set = count_sex.index
values_sex = count_sex.values

fig = plt.figure(figsize=(8, 5))
plt.bar(index_set, values_sex, color="maroon", width=0.5)
plt.xlabel("Jenis Kelamin")
plt.ylabel("Jumlah Penumpang")
plt.title("Chart Jumlah Penumpang Berdasarkan Jenis Kelamin")
for i, v in enumerate(values_sex):
    plt.text(i, v + 5, str(v), ha='center', fontweight='bold')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

In [ ]:
# Distribusi Keselamatan Berdasarkan Jenis Kelamin
plt.figure(figsize=(7, 5))
sns.countplot(data=data_titanic, x='Survived', hue='Sex', palette='viridis')
plt.title('Distribusi Keselamatan Berdasarkan Jenis Kelamin')
plt.xlabel('Status (0 = Tidak Selamat, 1 = Selamat)')
plt.ylabel('Jumlah Penumpang')
plt.legend(title='Jenis Kelamin')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

In [ ]:
# Probabilitas Tingkat Keselamatan Berdasarkan Jenis Kelamin
plt.figure(figsize=(6, 4))
ax = sns.barplot(x='Sex', y='Survived', data=data_titanic, palette='coolwarm')

# Label persentase pada bar
for p in ax.patches:
    value = p.get_height()
    ax.annotate(f'{value*100:.1f}%', 
                (p.get_x() + p.get_width() / 2., value / 2),
                ha='center', va='center', color='white', fontweight='bold', fontsize=11)

plt.title('Probabilitas Keselamatan Berdasarkan Jenis Kelamin')
plt.xlabel('Jenis Kelamin')
plt.ylabel('Peluang Selamat')
plt.tight_layout()
plt.show()

In [ ]:
# Tingkat Keselamatan Berdasarkan Kelas Tiket (Pclass)
plt.figure(figsize=(7, 5))
sns.countplot(data=data_titanic, x='Pclass', hue='Survived', palette='magma')
plt.title('Tingkat Keselamatan Berdasarkan Kelas Penumpang')
plt.xlabel('Kelas Penumpang (1 = Atas, 2 = Menengah, 3 = Bawah)')
plt.ylabel('Jumlah Penumpang')
plt.legend(title='Survived', labels=['Tidak Selamat', 'Selamat'])
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

In [ ]:
# Histogram Distribusi Umur Penumpang terhadap Keselamatan
plt.figure(figsize=(8, 5))
sns.histplot(data=data_titanic, x='Age', hue='Survived', kde=True, bins=30, palette='Set1')
plt.title('Distribusi Umur Terhadap Keselamatan Penumpang')
plt.xlabel('Umur (Tahun)')
plt.ylabel('Frekuensi')
plt.show()

In [ ]:
# Interaksi Pelabuhan Keberangkatan, Jenis Kelamin, dan Peluang Selamat (Multivariate)
plt.figure(figsize=(8, 5))
sns.pointplot(x='Embarked', y='Survived', hue='Sex', data=data_titanic, 
              markers=['o', 's'], linestyles=['-', '--'])
plt.title('Interaksi Jenis Kelamin, Pelabuhan Keberangkatan, dan Peluang Selamat')
plt.xlabel('Pelabuhan Keberangkatan (Embarked)')
plt.ylabel('Peluang Selamat (Survived)')
plt.grid(axis='y', alpha=0.3)
plt.show()

## 3. Penanganan Nilai Hilang (Missing Value Handling)

**Kriteria Penanganan Standar Data Sains:**
* Jika data hilang **< 5%**: Lakukan penghapusan baris (*drop rows*).
* Jika data hilang **5% – 30%**: Lakukan imputasi (*median* untuk numerik, *modus* untuk kategorik).
* Jika data hilang **> 50%**: Fitur sebaiknya didrop karena mengisi data yang mayoritas hilang berisiko menimbulkan bias/noise pada model.

In [ ]:
# Pemeriksaan persentase missing value pada Data Latih
print("--- Missing Values pada Data Latih (data_titanic) ---")
missing_train = pd.DataFrame({
    'Missing Count': data_titanic.isnull().sum(),
    'Percentage (%)': (data_titanic.isnull().sum() / len(data_titanic)) * 100
})
display(missing_train[missing_train['Missing Count'] > 0])

In [ ]:
# Pemeriksaan persentase missing value pada Data Prediksi
print("--- Missing Values pada Data Prediksi (data_prediksi) ---")
missing_test = pd.DataFrame({
    'Missing Count': data_prediksi.isnull().sum(),
    'Percentage (%)': (data_prediksi.isnull().sum() / len(data_prediksi)) * 100
})
display(missing_test[missing_test['Missing Count'] > 0])

### Eksekusi Imputasi dan Drop Fitur
* Kolom `Age` diimputasi dengan **median** data latih agar robust terhadap outlier.
* Kolom `Fare` di data_prediksi diimputasi dengan **median** data latih.
* Kolom `Embarked` diimputasi dengan **modus** ('S').
* Kolom `Cabin` didrop karena missing value > 70%.

In [ ]:
# Imputasi Nilai Numerik (Bebas dari Deprecated inplace=True)
median_age = data_titanic['Age'].median()
median_fare = data_titanic['Fare'].median()
modus_embarked = data_titanic['Embarked'].mode()[0]

# Imputasi pada data_titanic
data_titanic['Age'] = data_titanic['Age'].fillna(median_age)
data_titanic['Embarked'] = data_titanic['Embarked'].fillna(modus_embarked)

# Imputasi pada data_prediksi (menggunakan statistik dari data latih agar tidak terjadi data leakage)
data_prediksi['Age'] = data_prediksi['Age'].fillna(median_age)
data_prediksi['Fare'] = data_prediksi['Fare'].fillna(median_fare)
data_prediksi['Embarked'] = data_prediksi['Embarked'].fillna(modus_embarked)

# Drop kolom Cabin pada kedua dataset
data_titanic = data_titanic.drop(columns=['Cabin'])
data_prediksi = data_prediksi.drop(columns=['Cabin'])

print("Missing values setelah penanganan:")
print("data_titanic missing:", data_titanic.isnull().sum().sum())
print("data_prediksi missing:", data_prediksi.isnull().sum().sum())

## 4. Transformasi Data & Feature Engineering

In [ ]:
# 1. Binary Encoding variabel Sex
sex_mapping = {"male": 1, "female": 0}
data_titanic['Sex'] = data_titanic['Sex'].map(sex_mapping)
data_prediksi['Sex'] = data_prediksi['Sex'].map(sex_mapping)

# 2. One-Hot Encoding variabel Embarked
ohe_train = pd.get_dummies(data_titanic[['Embarked']], dtype=int)
data_titanic = pd.concat([data_titanic.drop(columns=['Embarked']), ohe_train], axis=1)

ohe_test = pd.get_dummies(data_prediksi[['Embarked']], dtype=int)
data_prediksi = pd.concat([data_prediksi.drop(columns=['Embarked']), ohe_test], axis=1)

# 3. Feature Engineering: Ekstraksi Title dari Nama (Gunakan raw string r'...' untuk cegah SyntaxWarning)
data_titanic['Title'] = data_titanic['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
data_prediksi['Title'] = data_prediksi['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)

# Mapping Title ke kategori numerik
title_mapping = {
    "Mr": 0, "Miss": 1, "Mrs": 2, 
    "Master": 3, "Dr": 3, "Rev": 3, "Col": 3, "Major": 3, 
    "Mlle": 3, "Countess": 3, "Ms": 3, "Lady": 3, "Sir": 3, 
    "Capt": 3, "Don": 3, "Mme": 3, "Jonkheer": 3, "Dona": 3
}

data_titanic['Title'] = data_titanic['Title'].map(title_mapping).fillna(3).astype(int)
data_prediksi['Title'] = data_prediksi['Title'].map(title_mapping).fillna(3).astype(int)

# 4. Drop variabel identitas unik yang tidak memiliki nilai prediktif
id_cols = ['PassengerId', 'Name', 'Ticket']
data_titanic = data_titanic.drop(columns=id_cols)
data_prediksi = data_prediksi.drop(columns=id_cols)

# Membersihkan spasi pada nama kolom
data_titanic.columns = data_titanic.columns.str.strip()
data_prediksi.columns = data_prediksi.columns.str.strip()

print("--- Struktur Data Latih Siap Model ---")
display(data_titanic.head())

## 5. Pembagian Data (Train-Test Split) & Standarisasi Fitur

In [ ]:
# Memisahkan Fitur (X) dan Target (y)
target = 'Survived'
X = data_titanic.drop(columns=[target])
y = data_titanic[target]

# Menyamakan urutan kolom data_prediksi dengan X
data_prediksi = data_prediksi.reindex(columns=X.columns, fill_value=0)

# Train-Test Split (80% Training, 20% Testing) dengan stratify=y
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Jumlah Data Latih (X_train): {len(X_train)}")
print(f"Jumlah Data Uji   (X_test) : {len(X_test)}")
print(f"Total Sampel                : {len(X_train) + len(X_test)}")

# Feature Scaling (StandardScaler) untuk Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_pred_scaled = scaler.transform(data_prediksi)

## 6. Pemodelan (Logistic Regression & Decision Tree)

In [ ]:
# 1. Model Logistic Regression
log_reg = LogisticRegression(random_state=42)
log_reg.fit(X_train_scaled, y_train)

# 2. Model Decision Tree (max_depth=3 untuk mencegah overfitting)
dt = DecisionTreeClassifier(criterion='gini', max_depth=3, random_state=42)
dt.fit(X_train, y_train)

# Visualisasi Pohon Keputusan (Decision Tree)
plt.figure(figsize=(20, 10))
plot_tree(
    dt, 
    feature_names=X_train.columns.tolist(), 
    class_names=['Tidak Selamat', 'Selamat'], 
    filled=True, 
    rounded=True, 
    fontsize=11
)
plt.title("Visualisasi Struktur Decision Tree (Max Depth = 3)", fontsize=16, fontweight='bold')
plt.show()

## 7. Evaluasi Komparatif Model

Model dievaluasi menggunakan data uji validasi (`X_test` dan `X_test_scaled`):
* `y_pred`: Hasil klasifikasi biner (0 atau 1) untuk menghitung Akurasi, Presisi, Recall, F1, dan Confusion Matrix.
* `y_prob`: Nilai probabilitas kontinu untuk menghitung Kurva ROC dan Skor AUC secara tepat.

In [ ]:
# Prediksi Label Biner
y_pred_log = log_reg.predict(X_test_scaled)
y_pred_dt = dt.predict(X_test)

# Prediksi Probabilitas Kelas 1 (PENTING untuk ROC-AUC yang halus dan presisi)
y_prob_log = log_reg.predict_proba(X_test_scaled)[:, 1]
y_prob_dt = dt.predict_proba(X_test)[:, 1]

models = [('Logistic Regression', y_pred_log), ('Decision Tree', y_pred_dt)]

# Visualisasi Confusion Matrix Komparatif
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for idx, (nama, y_pred) in enumerate(models):
    cm = confusion_matrix(y_test, y_pred)
    cmap = 'Blues' if nama == 'Logistic Regression' else 'Greens'
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, ax=axes[idx],
                xticklabels=['Tidak Selamat', 'Selamat'],
                yticklabels=['Tidak Selamat', 'Selamat'])
    axes[idx].set_title(f'Confusion Matrix: {nama}', fontweight='bold')
    axes[idx].set_xlabel('Prediksi Model')
    axes[idx].set_ylabel('Data Aktual')
plt.tight_layout()
plt.show()

In [ ]:
# Classification Report untuk kedua model
print("=== CLASSIFICATION REPORT: LOGISTIC REGRESSION ===")
print(classification_report(y_test, y_pred_log, target_names=['Tidak Selamat', 'Selamat']))

print("\n=== CLASSIFICATION REPORT: DECISION TREE ===")
print(classification_report(y_test, y_pred_dt, target_names=['Tidak Selamat', 'Selamat']))

### K-Fold Cross Validation (K=5) & Evaluasi Stabilitas Model

In [ ]:
# Setting 5-Fold Cross Validation
k_fold = KFold(n_splits=5, shuffle=True, random_state=42)

# Evaluasi Cross-Validation
score_log = cross_val_score(log_reg, X_train_scaled, y_train, cv=k_fold, scoring='accuracy')
score_dt = cross_val_score(dt, X_train, y_train, cv=k_fold, scoring='accuracy')

print(f"Rata-rata Akurasi K-Fold LogReg : {score_log.mean():.4f} (Std Dev: {score_log.std():.4f})")
print(f"Rata-rata Akurasi K-Fold Decision Tree : {score_dt.mean():.4f} (Std Dev: {score_dt.std():.4f})")

### Kurva ROC & Nilai AUC (Menggunakan Probabilitas Sebenarnya)

In [ ]:
# Menghitung False Positive Rate (FPR) dan True Positive Rate (TPR) berbasis PROBABILITAS
fpr_log, tpr_log, _ = roc_curve(y_test, y_prob_log)
roc_auc_log = auc(fpr_log, tpr_log)

fpr_dt, tpr_dt, _ = roc_curve(y_test, y_prob_dt)
roc_auc_dt = auc(fpr_dt, tpr_dt)

# Plot Kurva ROC
plt.figure(figsize=(8, 6))
plt.plot(fpr_log, tpr_log, color='royalblue', lw=2.5, 
         label=f'Logistic Regression (AUC = {roc_auc_log:.4f})')
plt.plot(fpr_dt, tpr_dt, color='darkorange', lw=2.5, 
         label=f'Decision Tree (AUC = {roc_auc_dt:.4f})')
plt.plot([0, 1], [0, 1], color='gray', lw=1.5, linestyle='--', label='Garis Acak / Baseline (AUC = 0.50)')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=11)
plt.ylabel('True Positive Rate (Recall / Sensitivity)', fontsize=11)
plt.title('Kurva ROC Komparatif: Logistic Regression vs Decision Tree', fontsize=13, fontweight='bold')
plt.legend(loc="lower right", fontsize=11)
plt.grid(alpha=0.3)
plt.show()

## 8. Tabel Rangkuman Performa & Evaluasi Metrik Model (Key Highlights)

Tabel di bawah ini merangkum seluruh parameter evaluasi kinerja kedua model klasifikasi untuk menentukan model terbaik secara objektif:

In [ ]:
# Rangkuman Seluruh Metrik Evaluasi dalam 1 Tabel Komparatif
eval_summary = pd.DataFrame({
    'Model': ['Logistic Regression', 'Decision Tree (max_depth=3)'],
    'Accuracy': [accuracy_score(y_test, y_pred_log), accuracy_score(y_test, y_pred_dt)],
    'Precision': [precision_score(y_test, y_pred_log), precision_score(y_test, y_pred_dt)],
    'Recall': [recall_score(y_test, y_pred_log), recall_score(y_test, y_pred_dt)],
    'F1-Score': [f1_score(y_test, y_pred_log), f1_score(y_test, y_pred_dt)],
    'ROC-AUC Score': [roc_auc_log, roc_auc_dt],
    'K-Fold Mean Accuracy': [score_log.mean(), score_dt.mean()],
    'K-Fold Std (Stabilitas)': [score_log.std(), score_dt.std()]
})

# Format tampilan agar mudah dianalisis
print("--- TABEL PERBANDINGAN PERFORMA MODEL KLASIFIKASI ---")
display(eval_summary.style.format({
    'Accuracy': '{:.4f}',
    'Precision': '{:.4f}',
    'Recall': '{:.4f}',
    'F1-Score': '{:.4f}',
    'ROC-AUC Score': '{:.4f}',
    'K-Fold Mean Accuracy': '{:.4f}',
    'K-Fold Std (Stabilitas)': '{:.4f}'
}).set_properties(**{'text-align': 'center'}))

### Analisis Kepentingan Fitur (*Feature Importance*) Model Terbaik (Decision Tree)

In [ ]:
# Feature Importance dari Model Decision Tree
feat_importances = pd.Series(dt.feature_importances_, index=X_train.columns).sort_values(ascending=True)

plt.figure(figsize=(9, 5))
feat_importances.plot(kind='barh', color='teal', edgecolor='black')
plt.title('Tingkat Kepentingan Variabel Prediktor (Feature Importance - Decision Tree)', fontsize=12, fontweight='bold')
plt.xlabel('Gini Importance (Skor Kepentingan Relatif)')
plt.ylabel('Variabel Fitur')
plt.grid(axis='x', linestyle='--', alpha=0.6)
for index, value in enumerate(feat_importances):
    plt.text(value + 0.01, index, f"{value:.4f}", va='center', fontweight='bold')
plt.tight_layout()
plt.show()

### **Kesimpulan Evaluasi Model:**
1. **Decision Tree (max_depth=3)** terpilih sebagai model terbaik karena mengungguli Logistic Regression di metrik utama:
   * **Akurasi Uji:** 82,12% (vs 81,01%)
   * **Recall (Sensitivitas Deteksi Selamat):** 76,81% (vs 69,57%)
   * **F1-Score:** 0,7681 (vs 0,7385)
   * **Stabilitas K-Fold:** Standar deviasi Decision Tree lebih kecil (**0,0397** vs 0,0496), menandakan performa model lebih konsisten di berbagai lipatan data dan tidak overfit.
2. Variabel **Title (gelar sosial)**, **Sex (jenis kelamin)**, dan **Pclass (kelas kabin)** terbukti secara empiris sebagai 3 fitur paling dominan dalam memprediksi probabilitas keselamatan penumpang.

## 9. Penerapan Model Terbaik pada Data Prediksi (Data Baru)

In [ ]:
# Prediksi pada Data Uji Mandiri (data_prediksi) menggunakan Decision Tree
pred_dt = dt.predict(data_prediksi)
data_prediksi['Prediksi_Survived'] = pred_dt

print("Jumlah Hasil Prediksi Keselamatan:")
print(data_prediksi['Prediksi_Survived'].value_counts())

# Simpan hasil prediksi ke file CSV dan Excel
data_prediksi.to_csv('hasil_prediksi.csv', index=False)
data_prediksi.to_excel('hasil_prediksi.xlsx', index=False)
print("\nFile 'hasil_prediksi.csv' dan 'hasil_prediksi.xlsx' berhasil disimpan!")

In [ ]:
# Visualisasi Proporsi Hasil Prediksi
counts = data_prediksi['Prediksi_Survived'].value_counts()

plt.figure(figsize=(6, 6))
plt.pie(counts, labels=['Tidak Selamat', 'Selamat'], autopct='%1.1f%%', 
        colors=['salmon', 'lightblue'], startangle=90, explode=(0.05, 0), shadow=True)
plt.title('Proporsi Hasil Prediksi Keselamatan (Decision Tree)', fontweight='bold')
plt.show()

In [ ]:
# Heatmap Korelasi Fitur Terhadap Hasil Prediksi
corr_pred = data_prediksi.corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr_pred, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Heatmap Korelasi Fitur dan Hasil Prediksi', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Crosstab Analisis Gender vs Hasil Prediksi
sex_survived = pd.crosstab(
    data_prediksi['Sex'],
    data_prediksi['Prediksi_Survived'],
    margins=True,
    margins_name='Total'
)
sex_survived.index = ['Perempuan (0)', 'Laki-Laki (1)', 'Total']
sex_survived.columns = ['Tidak Selamat (0)', 'Selamat (1)', 'Total']

print("--- Tabel Silang Jenis Kelamin vs Hasil Prediksi ---")
display(sex_survived)